In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from textwrap import wrap

In [ ]:
artifacts_dir = Path("artifacts_paper")
baseline_acc = artifacts_dir / "baseline_acc.json"
embedding_acc = artifacts_dir / "embedding_acc.json"
assert baseline_acc.exists() and embedding_acc.exists(), (
    "The artifacts directory does not contain the expected files."
)

In [ ]:
baseline_df = pd.read_json(baseline_acc, orient="records", lines=True)
embedding_df = pd.read_json(embedding_acc, orient="records", lines=True)

In [ ]:
baseline_df = pd.read_json(baseline_acc, orient="records", lines=True)
embedding_df = pd.read_json(embedding_acc, orient="records", lines=True)
embedding_df.dropna(subset=["embedding_performance"], inplace=True)
embedding_pivot = embedding_df.pivot(
    index="topic_title", columns="which", values="embedding_performance"
)

acc_df = embedding_pivot.merge(
    baseline_df[["topic_title", "baseline_performance"]], on="topic_title"
)
avg_acc = acc_df.set_index("topic_title").mean(axis=0).sort_values(ascending=False)
mean_series = acc_df.set_index("topic_title").mean(axis=0)
std_series = acc_df.set_index("topic_title").std(axis=0)

acc_stats_df = pd.DataFrame({
    "Accuracy": mean_series,
    "Std": std_series
})

acc_stats_df["Accuracy ± Std"] = acc_stats_df.apply(
    lambda row: f"{row['Accuracy']:.2f} ± {row['Std']:.2f}", axis=1
)

acc_stats_df = acc_stats_df.sort_values(by="Accuracy", ascending=False)

acc_df.set_index("topic_title").to_excel(
    artifacts_dir / "acc.xlsx",
    index=True,
    float_format="%.2f",
    sheet_name="accuract_per_topic",
)
area_df = embedding_df.merge(
    baseline_df[["topic_title", "baseline_performance"]], on="topic_title"
)
line_plot_df = (
    area_df.dropna()
    .groupby(["course_title", "which"])
    .agg({"embedding_performance": "mean"})
    .rename(columns={"embedding_performance": "performance"})
    .reset_index()
)

baseline_df = (
    area_df.dropna()
    .groupby("course_title")["baseline_performance"]
    .mean()
    .reset_index()
    .rename(columns={"baseline_performance": "performance"})
)
baseline_df["which"] = "Baseline"

# Combine both DataFrames
final_df = pd.concat([line_plot_df, baseline_df], ignore_index=True)

final_df.rename(
    columns={"course_title": "Area", "which": "Method", "performance": "Performance"},
    inplace=True,
)
final_df["Area"] = [
    "\n".join(wrap(x, 12)) for x in final_df["Area"]
]


In [ ]:
rc_params = {
    "axes.spines.right": False,
    "axes.spines.top": False,
    "grid.linestyle": "--",
    "grid.linewidth": 0.5,
    "font.family": "serif",
}
sns.set_theme(style="ticks", rc=rc_params, font_scale=1)
hue_labels = avg_acc.index.to_list()
hue_labels[hue_labels.index("baseline_performance")] = "Baseline"

fig, ax = plt.subplots(figsize=(15, 8))

# sort the hu_labels based on the 
ax = sns.lineplot(
    data=final_df,
    x="Area",
    y="Performance",
    hue="Method",
    style="Method",
    errorbar=None,
    markeredgecolor="black",
    palette="tab10",
    markers=True,
    dashes=False,
    markersize=10,
    alpha=0.8,
    lw=3,
    hue_order=hue_labels,
)

ax.set_xlabel(None)
ax.set_ylabel("Accuracy", fontweight="bold")
# set xticks bold
ax.set_xticks(ax.get_xticks(), ax.get_xticklabels(), fontweight="bold")
ax.set_yticks(ax.get_yticks(), ax.get_yticklabels(), fontweight="bold")

# # rotate x labels by 45
# plt.xticks(rotation=90);
plt.grid(True)
# plt.title("Model Performance Comparison")
# plt.rcParams.update({'font.size': 22})
plt.legend(fontsize="small")
sns.move_legend(ax, "upper center", bbox_to_anchor=(0.5, -0.15), ncols=3)
# set title
fig.suptitle(
    "Model Performance Comparison",
    fontsize=20,
    fontweight="bold",

)
plt.tight_layout()
plt.savefig(
    artifacts_dir / "model_performance_comparison.pdf",
    dpi=300,
    bbox_inches="tight",
)

In [ ]:
# Precision at k
baseline_precision_at_3 = artifacts_dir / "precision_baseline_at_3.json"
embedding_precision_at_3 = artifacts_dir / "precision_embedding_at_3.json"
baseline_precision_at_5 = artifacts_dir / "precision_baseline_at_5.json"
embedding_precision_at_5 = artifacts_dir / "precision_embedding_at_5.json"
assert (
    baseline_precision_at_3.exists()
    and baseline_precision_at_5.exists()
    and embedding_precision_at_3.exists()
    and embedding_precision_at_5.exists()
), "The artifacts directory does not contain the expected files."

In [ ]:
baseline_precision_at_3_df = pd.read_json(
    baseline_precision_at_3, orient="records", lines=True
)

baseline_precision_at_5_df = pd.read_json(
    baseline_precision_at_5, orient="records", lines=True
)
embedding_precision_at_5_df = pd.read_json(
    embedding_precision_at_5, orient="records", lines=True
)
embedding_precision_at_3_df = pd.read_json(
    embedding_precision_at_3, orient="records", lines=True
)
baseline_precision = baseline_precision_at_3_df.merge(baseline_precision_at_5_df, on=["topic_title", "course_title"])
baseline_precision["which"] = "Baseline"
baseline_precision.rename(columns={"baseline_precision_at_3": "Precision@3", "baseline_precision_at_5": "Precision@5"}, inplace=True)

embedding_precision = embedding_precision_at_3_df.merge(embedding_precision_at_5_df, on=["topic_title", "course_title", "which"])
embedding_precision.rename(columns={"emb_precision_at_3": "Precision@3", "emb_precision_at_5": "Precision@5"}, inplace=True)
precision_df = pd.concat([baseline_precision, embedding_precision], ignore_index=True)


In [ ]:
# groupby precision_df by course_title and then write each group to a separate excel sheet
# rename which to Method
precision_df.rename(columns={"which": "Method"}, inplace=True)
with pd.ExcelWriter(artifacts_dir / "precision.xlsx") as writer:
    for course_title, group in precision_df.groupby("course_title"):
        group.to_excel(writer, sheet_name=course_title[:30], index=False, float_format="%.2f")